# Week 3 &mdash; Message Passing and Spatial GNNs

## Implementation: MPNN from Scratch, then GCN and GAT in PyTorch Geometric

This notebook (i) implements a message-passing layer in pure NumPy to expose the mechanics, (ii) verifies permutation equivariance, and (iii) trains a **GCN** and a **GAT** on a node-classification benchmark using **PyTorch Geometric**.

### Setup


In [ ]:
# Core dependencies:
# !pip install numpy torch
# PyTorch Geometric (choose the wheel matching your torch/CUDA version):
# !pip install torch_geometric

import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)


## 1. A Message-Passing Layer in Pure NumPy

We implement one MPNN layer with a *sum* aggregator and a linear message function, mirroring the framework from the theory notebook:
$$
m_{ij} = W h_j, \qquad a_i = \sum_{j \in \mathcal{N}(i)} m_{ij}, \qquad h_i' = \sigma(W h_i + a_i).
$$


In [ ]:
def relu(z):
    return np.maximum(z, 0.0)

def mpnn_layer(H, A, W_self, W_msg):
    """One message-passing layer with sum aggregation.

    H        : (n, d_in)  node features
    A        : (n, n)     adjacency (no self loops)
    W_self   : (d_in, d_out)
    W_msg    : (d_in, d_out)
    """
    messages = H @ W_msg            # (n, d_out): message from each node
    aggregate = A @ messages        # sum over neighbours
    return relu(H @ W_self + aggregate)

n, d_in, d_out = 6, 4, 3
A = np.array([
    [0,1,1,0,0,0],
    [1,0,1,0,0,0],
    [1,1,0,1,0,0],
    [0,0,1,0,1,1],
    [0,0,0,1,0,1],
    [0,0,0,1,1,0],
], dtype=float)
H = rng.normal(size=(n, d_in))
W_self = rng.normal(size=(d_in, d_out))
W_msg = rng.normal(size=(d_in, d_out))

out = mpnn_layer(H, A, W_self, W_msg)
print("Output node features:\n", out)


## 2. Verifying Permutation Equivariance

We apply a random permutation $P$ to both the features ($PH$) and the adjacency ($PAP^\top$), and check that the layer output is the correspondingly permuted version of the original output.


In [ ]:
perm = rng.permutation(n)
P = np.eye(n)[perm]

out_perm = mpnn_layer(P @ H, P @ A @ P.T, W_self, W_msg)   # f(PH, PAP^T)
out_ref = P @ out                                          # P f(H, A)

print("Permutation-equivariance error:", np.abs(out_perm - out_ref).max())


The error is at floating-point precision: the layer is exactly permutation-equivariant, confirming the proposition from the theory notebook.


## 3. Implementing the GCN Normalisation

Next we implement the GCN's symmetric renormalisation $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ with $\tilde{A} = A + I$, and use it as the aggregation operator.


In [ ]:
def gcn_norm(A):
    A_tilde = A + np.eye(A.shape[0])           # add self-loops
    d = A_tilde.sum(axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return d_inv_sqrt @ A_tilde @ d_inv_sqrt

A_hat = gcn_norm(A)
print("Normalised adjacency A_hat (symmetric, spectral radius <= 1):\n", A_hat)
print("\nIs symmetric?", np.allclose(A_hat, A_hat.T))
print("Spectral radius:", round(np.abs(np.linalg.eigvalsh(A_hat)).max(), 4))


## 4. GCN for Node Classification with PyTorch Geometric

We now move to a realistic setting. The following cell defines and trains a two-layer GCN on the **Cora** citation network, a standard semi-supervised node-classification benchmark. Run it in an environment with `torch` and `torch_geometric` installed.


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]
print(data)   # Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], ...)

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCN(dataset.num_features, 16, dataset.num_classes).to(device)
data = data.to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train_step():
    model.train(); opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); opt.step()
    return float(loss)

@torch.no_grad()
def test():
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=1)
    accs = []
    for mask in (data.train_mask, data.val_mask, data.test_mask):
        accs.append(float((pred[mask] == data.y[mask]).float().mean()))
    return accs

for epoch in range(1, 201):
    loss = train_step()
    if epoch % 40 == 0:
        tr, va, te = test()
        print(f"Epoch {epoch:3d} | loss {loss:.4f} | train {tr:.3f} | val {va:.3f} | test {te:.3f}")

print("\nFinal test accuracy: {:.3f}".format(test()[2]))


## 5. GAT on the Same Benchmark

We swap the GCN convolutions for **multi-head graph attention** layers. Note the use of `heads` in the first layer (concatenated) and a single averaging head in the output layer.


In [ ]:
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=0.6)
        # Concatenated heads -> hidden * heads input to the next layer.
        self.conv2 = GATConv(hidden * heads, out_dim, heads=1,
                             concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        return self.conv2(x, edge_index)

model = GAT(dataset.num_features, 8, dataset.num_classes, heads=8).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

for epoch in range(1, 201):
    loss = train_step()
    if epoch % 40 == 0:
        tr, va, te = test()
        print(f"Epoch {epoch:3d} | loss {loss:.4f} | train {tr:.3f} | val {va:.3f} | test {te:.3f}")

print("\nFinal GAT test accuracy: {:.3f}".format(test()[2]))


## 6. Discussion: GCN vs. GAT

On Cora both models typically reach **80&ndash;83%** test accuracy. GAT often edges ahead on graphs where neighbours vary in informativeness, because it *learns* per-edge weights rather than relying on fixed degree normalisation. The trade-off is more parameters and higher computational cost per layer (one attention score per edge per head).

This completes the spatial-GNN core of the course. In Week 4 we leave the abstract-graph setting and apply these ideas to **3D point clouds**, where the relevant symmetry is the rigid-motion group $SE(3)$ rather than mere permutations.


## 7. Exercises

1. Replace the sum aggregator in `mpnn_layer` with a **mean** aggregator and re-verify permutation equivariance.
2. Implement the **GIN** update $h_i' = \mathrm{MLP}\big((1+\epsilon)h_i + \sum_j h_j\big)$ in NumPy and discuss why the sum makes it more expressive than the mean.
3. Add a third GCN/GAT layer. Does accuracy improve or degrade? Relate your observation to *over-smoothing* in deep GNNs.
